# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vikraamkumar-ds/flyrank-internship-ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [21]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/vikraamkumar-ds/flyrank-internship-ml"
REPO_DIR = "flyrank-internship-ml"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found -- are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-internship-ml/flyrank-internship-ml/flyrank-internship-ml
Starter data found. You're ready.


In [22]:
import pandas as pd
df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
print(df.shape)
df.head()

(30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [23]:
for i, col in enumerate(df.columns):
    print(i, col)

0 content_id
1 client_id
2 search_volume
3 competition
4 competition_level
5 cpc
6 content_type
7 main_intent
8 word_count
9 char_count
10 provider_used
11 model_used
12 impressions_90d
13 clicks_90d
14 pageviews_90d
15 sessions_90d
16 users_90d
17 engaged_sessions_90d
18 ai_sessions_90d
19 scroll_events_90d
20 days_with_impressions
21 days_with_sessions
22 impressions_last_30d
23 clicks_last_30d
24 sessions_last_30d
25 impressions_prev_30d
26 clicks_prev_30d
27 sessions_prev_30d
28 content_age_days
29 age_tier
30 age_tier_order
31 days_since_last_update
32 freshness_tier
33 word_count_tier
34 char_count_tier
35 ctr
36 avg_position
37 engagement_rate
38 scroll_rate
39 ai_traffic_pct
40 impression_tier
41 position_tier
42 trend_direction
43 trend_pct


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Cell A (code) — build the label and Signal 1 (staleness):

In [24]:
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

df['staleness_bucket'] = pd.cut(
    df['days_since_last_update'],
    bins=[-1, 90, 180, 365, 100000],
    labels=['<90d', '90-180d', '180-365d', '365d+']
)

staleness_stats = df.groupby('staleness_bucket', observed=True)['is_declining_label'].agg(['mean','count']).reset_index()
staleness_stats.columns = ['bucket', 'decline_rate', 'n']
print(staleness_stats)

     bucket  decline_rate      n
0      <90d      0.512031  20655
1   90-180d      0.611057   9171
2  180-365d      0.467456    169
3     365d+      0.600000      5


Cell B (code) — Signal 2 (CTR vs. position):

In [25]:
# expected CTR per position tier (group average), then how far each page is below it
tier_avg_ctr = df.groupby('position_tier', observed=True)['ctr'].transform('mean')
df['ctr_gap'] = df['ctr'] - tier_avg_ctr

df['ctr_gap_bucket'] = pd.qcut(df['ctr_gap'], q=4, labels=['worst','below_avg','above_avg','best'])

ctr_stats = df.groupby('ctr_gap_bucket', observed=True)['is_declining_label'].agg(['mean','count']).reset_index()
ctr_stats.columns = ['bucket', 'decline_rate', 'n']
print(ctr_stats)

      bucket  decline_rate      n
0      worst      0.497819   7567
1  below_avg      0.580666  10655
2  above_avg      0.557863   4286
3       best      0.522824   7492


In [26]:
print(ctr_stats)

      bucket  decline_rate      n
0      worst      0.497819   7567
1  below_avg      0.580666  10655
2  above_avg      0.557863   4286
3       best      0.522824   7492


I picked two signals that back real FlyRank flags: how stale a page is (feeds the refresh
flags) and how a page's CTR compares to others at the same position (feeds the CTR-fix logic).

Staleness (days_since_last_update): I bucketed pages into <90d, 90-180d, 180-365d, 365d+ and
looked at how often each bucket actually has a declining trend. Decline rate was 51.2% for
<90d (n=20,655), jumped to 61.1% for 90-180d (n=9,171), then dropped back to 46.7% for
180-365d (n=169) and 60.0% for 365d+ (n=5). That's not a clean "staler = more likely
declining" story, and the last bucket is basically 5 rows so I don't trust it at all.
Verdict: MIXED.

CTR vs. position peers: for each page I compared its CTR to the average CTR of others in the
same position_tier, then split that gap into quartiles. I expected the worst-gap pages
(underperforming their peers) to decline the most. Instead the worst bucket had the *lowest*
decline rate (49.8%, n=7,567), and the two middle buckets were highest. That's the opposite
of what I expected. Verdict: OPPOSITE.

Since neither signal cleanly predicts decline on its own, I'm not building a "predict decline"
rule — I'm building a triage rule: which pages are worth an editor's time to check, based on
staleness plus demand, not based on a claim that these signals cause decline.

Rule (plain words): flag a page for review if it hasn't been touched in 90+ days AND its
search demand is at or above the median — old content that's actually worth fixing, not a
dead page nobody searches for.

Reason codes: STALE_HIGH_DEMAND, STALE_ONLY, HIGH_DEMAND_ONLY, NO_SIGNAL

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [27]:
vol_median = df['search_volume'].median()

def score_row(row):
    score = 0
    reasons = []
    if row['days_since_last_update'] > 90:
        score += 2
        reasons.append(('STALE_CONTENT', 2))
    if row['search_volume'] >= vol_median:
        score += 1
        reasons.append(('HIGH_DEMAND', 1))
    if not reasons:
        return pd.Series([0, 'NO_SIGNAL'])
    reasons.sort(key=lambda x: -x[1])
    top_reason = reasons[0][0]
    if len(reasons) == 2:
        top_reason = 'STALE_HIGH_DEMAND'
    return pd.Series([score, top_reason])

df[['rule_score', 'reason_code']] = df.apply(score_row, axis=1)

def action_label(score):
    if score >= 3:
        return 'REVIEW_NOW'
    elif score >= 1:
        return 'MONITOR'
    return 'NO_ACTION'

df['action'] = df['rule_score'].apply(action_label)

queue = df.sort_values('rule_score', ascending=False).reset_index(drop=True)

import os
os.makedirs('work/outputs', exist_ok=True)
queue.to_csv('work/outputs/baseline_action_score.csv', index=False)

print(queue['action'].value_counts())
print(queue[['content_id','rule_score','reason_code','action']].head(10))

action
MONITOR       16010
NO_ACTION      9097
REVIEW_NOW     4893
Name: count, dtype: int64
             content_id  rule_score        reason_code      action
0  content_d8ee6cc6d642           3  STALE_HIGH_DEMAND  REVIEW_NOW
1  content_30aac2f74fd3           3  STALE_HIGH_DEMAND  REVIEW_NOW
2  content_198bc46d31f5           3  STALE_HIGH_DEMAND  REVIEW_NOW
3  content_7664a2c4d132           3  STALE_HIGH_DEMAND  REVIEW_NOW
4  content_08109d938389           3  STALE_HIGH_DEMAND  REVIEW_NOW
5  content_2c5172b87af6           3  STALE_HIGH_DEMAND  REVIEW_NOW
6  content_e79d1cd54af1           3  STALE_HIGH_DEMAND  REVIEW_NOW
7  content_77867ed726e1           3  STALE_HIGH_DEMAND  REVIEW_NOW
8  content_4595e8704e07           3  STALE_HIGH_DEMAND  REVIEW_NOW
9  content_4df0b7207fe3           3  STALE_HIGH_DEMAND  REVIEW_NOW


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [28]:
top20 = queue.head(20)[['content_id', 'days_since_last_update', 'search_volume',
                          'avg_position', 'ctr', 'freshness_tier',
                          'rule_score', 'reason_code', 'action']]
top20

,content_id,days_since_last_update,search_volume,avg_position,ctr,freshness_tier,rule_score,reason_code,action
0,content_d8ee6cc6d642,104,20.0,2.2,1.55,91-180,3,STALE_HIGH_DEMAND,REVIEW_NOW
1,content_30aac2f74fd3,104,40.0,7.6,0.00,91-180,3,STALE_HIGH_DEMAND,REVIEW_NOW
2,content_198bc46d31f5,104,50.0,3.3,0.30,91-180,3,STALE_HIGH_DEMAND,REVIEW_NOW
3,content_7664a2c4d132,104,20.0,5.7,0.34,91-180,3,STALE_HIGH_DEMAND,REVIEW_NOW
4,content_08109d938389,104,40.0,11.2,0.23,91-180,3,STALE_HIGH_DEMAND,REVIEW_NOW
5,content_2c5172b87af6,104,10.0,26.2,0.00,91-180,3,STALE_HIGH_DEMAND,REVIEW_NOW
6,content_e79d1cd54af1,104,10.0,15.0,0.66,91-180,3,STALE_HIGH_DEMAND,REVIEW_NOW
7,content_77867ed726e1,104,30.0,12.7,0.08,91-180,3,STALE_HIGH_DEMAND,REVIEW_NOW
8,content_4595e8704e07,104,90.0,36.3,0.00,91-180,3,STALE_HIGH_DEMAND,REVIEW_NOW
9,content_4df0b7207fe3,151,10.0,49.6,1.65,91-180,3,STALE_HIGH_DEMAND,REVIEW_NOW


All 20 top rows land at score 3 (STALE_HIGH_DEMAND / REVIEW_NOW) — staleness is basically
identical across them (92-151 days), so search_volume and how well they're already ranking
are what actually separate a good pick from a questionable one.

1. d8ee6cc6d642 — vol 20, already ranks well (pos 2.2, CTR 1.55%). Wrong if it's a seasonal
   dip, not a real problem — this is probably my weakest pick.
2. 30aac2f74fd3 — vol 40, decent position (7.6) but 0% CTR. Wrong if that 0% is a tracking
   gap rather than an actual content issue.
3. 198bc46d31f5 — vol 50, mid position (3.3), CTR only 0.30%. Wrong if this keyword just has
   a naturally low CTR (e.g. an answer box eating clicks).
4. 7664a2c4d132 — vol 20, position 5.7, CTR 0.34%. Wrong if this traffic is low-intent anyway.
5. 08109d938389 — vol 40, position 11.2, CTR 0.23%. Wrong if the page recently moved and
   just hasn't settled yet.
6. 2c5172b87af6 — vol 10, weak position (26.2), 0% CTR. Wrong if volume of 10 isn't enough
   to justify an editor's time regardless of the other numbers.
7. e79d1cd54af1 — vol 10, position 15.0, CTR 0.66%. Wrong if editor time is better spent on
   higher-volume pages.
8. 77867ed726e1 — vol 30, position 12.7, CTR just 0.08%. Probably my strongest pick — decent
   position, almost no clicks.
9. 4595e8704e07 — vol 90 (top demand in this set), position 36.3, 0% CTR. Wrong if a page-4
   ranking reflects a relevance problem a refresh won't fix.
10. 4df0b7207fe3 — staler than the rest (151d), vol 10, weak position (49.6) but CTR 1.65%.
    Wrong if that CTR is just noise from very few impressions.
11. e89c41dd69ad — vol 10, position 16.6, 0% CTR. Wrong if low volume means it's not worth
    the time.
12. 6e4e8750dc29 — vol 10, position 15.1, CTR 0.16%. Same marginal case as #11.
13. 7b3e020817ae — only 2 days past the 90-day cutoff, vol 20, 0% CTR. Wrong if it's really
    "fresh enough" and the threshold is just catching an edge case.
14. 1d4d78ba6371 — vol 10, position 7.1, 0% CTR. Wrong if bandwidth should go to higher-volume
    pages first.
15. 9b6fe1b0aa1a — vol 30, position 24.4, CTR 0.10%. Reasonable case — weak position, low CTR.
16. 4e8fa9dda799 — vol 50, position 33.8, 0% CTR. Good candidate — page 4, no clicks at all.
17. 672947e34836 — highest volume in the set (110), position 45.5, 0% CTR. Biggest upside if
    it's fixable; wrong if position 45 means a relevance mismatch, not just staleness.
18. d8bc8fbfca08 — vol 50, position 36.1, 0% CTR. Similar case to #16.
19. eb28cfe1c540 — vol 10, position 24.2, CTR 0.22%. Marginal, low volume.
20. 7eee5a3c2c5a — vol 10, position 16.2, CTR 1.11%. Wrong if this page is already fine and
    doesn't need action.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [29]:
rule_inputs = ['days_since_last_update', 'search_volume']
leaky_cols = ['trend_direction', 'trend_pct', 'impressions_last_30d', 'clicks_last_30d',
              'impressions_prev_30d', 'clicks_prev_30d']
print("Rule inputs used:", rule_inputs)
print("Confirmed NOT used in rule:", [c for c in leaky_cols if c not in rule_inputs])

Rule inputs used: ['days_since_last_update', 'search_volume']
Confirmed NOT used in rule: ['trend_direction', 'trend_pct', 'impressions_last_30d', 'clicks_last_30d', 'impressions_prev_30d', 'clicks_prev_30d']


Weakest picks: #1 and #20 both already rank decently (positions 2.2 and 16.2) with non-zero
CTR — the rule flagged them purely for being stale + high-demand, without checking whether
they're actually underperforming. #13 is also weak: it's only 2 days past the 90-day cutoff,
which shows the threshold creates arbitrary edge cases rather than a real distinction.

Leakage check: the rule only uses days_since_last_update and search_volume. It doesn't touch
trend_direction, trend_pct, or any of the _last_30d/_prev_30d columns — those are what the
label (and its likely trend calculation) are built from, so keeping them out avoids leakage.
No future-window or outcome-derived columns are in the rule.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.